In [17]:
from pathlib import Path
import sys
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
repo_root

WindowsPath('C:/Users/leungk/OneDrive - EllisDon Corporation/Documents/Other/github_repos/CBG_analysis')

# USDA Cache and Enrichment
Call FoodData Central, populate the local nutrient cache, and check misses so meal items get macros for glucose impact analysis.

In [18]:
from pathlib import Path
import pandas as pd
from src import io_excel, parse_foods, defaults, aggregate, usda_client

repo_root = Path('..').resolve()
excel_path = repo_root / 'data' / 'source_data' / '20251218_Trudy_Meals.xlsx'
defaults_path = repo_root / 'config' / 'defaults_food_items.yaml'
api_keys_path = repo_root / 'config' / 'api_keys.json'
cache_path = repo_root / 'notebook' / 'cache_data' / 'parquet' / 'food_nutrition_cache.parquet'

print(f'Loading clean events from {excel_path}')
clean_events = io_excel.load_clean_events(excel_path)
print(f'Clean events: {len(clean_events)} rows')

Loading clean events from C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\data\source_data\20251218_Trudy_Meals.xlsx
Clean events: 206 rows


C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\src\io_excel.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_parsed = pd.to_datetime(time_series, errors="coerce").dt.time


In [19]:
# Parse foods and apply defaults
items = parse_foods.explode_food_items(clean_events)
print(f'Exploded to items: {len(items)} rows')

cfg = defaults.load_defaults_config(defaults_path)
items = defaults.apply_default_rules(items, cfg)
assumed_rate = items['assumed_100g_flag'].mean() if 'assumed_100g_flag' in items else 0.0
print(f'Defaults applied; assumed_100g_flag rate: {assumed_rate:.3f}')

Exploded to items: 644 rows
Defaults applied; assumed_100g_flag rate: 0.000


In [20]:
# Convert to grams
items = aggregate.compute_grams(items)
print('Converted to grams; grams_final stats:')
print(items['grams_final'].describe())

Converted to grams; grams_final stats:
count    644.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: grams_final, dtype: float64


In [21]:
# Enrich with USDA cache/API
items, cache = usda_client.enrich_items_with_usda(items, api_keys_path, cache_path)
print(f'USDA cache entries: {len(cache)}')
print('usda_match_status counts:')
print(items['usda_match_status'].value_counts(dropna=False))
print('Sample matches/misses:')
print(items[['food_text_raw', 'food_name_std', 'usda_match_status']].head(15))

USDA cache entries: 189
usda_match_status counts:
usda_match_status
cache_or_api    491
missing         153
Name: count, dtype: int64
Sample matches/misses:
                 food_text_raw               food_name_std usda_match_status
0                       2 eggs                      2 eggs      cache_or_api
1               1.5 oz of beef              1 5 oz of beef      cache_or_api
2              2 cups choy sum             2 cups choy sum           missing
3                      fasting                         NaN           missing
4                 1 cup quinoa                1 cup quinoa      cache_or_api
5                       1 kiwi                      1 kiwi      cache_or_api
6      avocado oil for cooking     avocado oil for cooking      cache_or_api
7          2 cups chicken soup         2 cups chicken soup      cache_or_api
8          1.5 piece pork chop         1 5 piece pork chop           missing
9   1 cup stir fry green beans  1 cup stir fry green beans      cache_or_

C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\src\usda_client.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  "usda_description": result.get("description"),


In [22]:
# Items needing manual review
missing = items[items['usda_match_status'] == 'missing'][['food_text_raw', 'food_name_std']]
print(f'Missing USDA matches: {len(missing)}')
print(missing.head(30))

Missing USDA matches: 153
                             food_text_raw          food_name_std
2                          2 cups choy sum        2 cups choy sum
3                                  fasting                    NaN
8                      1.5 piece pork chop    1 5 piece pork chop
11                                 fasting                    NaN
12                                    none                   none
14                       1 piece pork chop      1 piece pork chop
15                            2 large eggs           2 large eggs
18                                    none                   none
23                     0.5 cup blueberries    0 5 cup blueberries
27                       1.5 cups choy sum      1 5 cups choy sum
28                                 fasting                    NaN
30                       3 oz. sliced beef       3 oz sliced beef
31                       2 cups watercress      2 cups watercress
43                         2 cups choy sum        